[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skous2/is455-w2027/blob/main/module_04/Worksheet_4B_Getting_and_Cleaning_Data.ipynb)

# Worksheet 4B · Getting and cleaning data

**Required, graded on completion. Due Wednesday, Feb 3 at 7:00 PM.** These are the same problems as Worksheet 4A, applied to Peakview Streaming. Expect about 40–50 minutes. Make an honest attempt at every question; wrong answers still earn full credit.

**Before you start,** choose **File → Save a copy in Drive** so your work is saved. Write every answer in your own words.

Readings for this module are the [requests Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/), McKinney 6.3 and chapter 7 ([wesmckinney.com/book](https://wesmckinney.com/book/)), and the [pandas documentation for `json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html). Worksheet 4A links each reading next to the section it covers.

## Where the data comes from

The clean Peakview Streaming file you've used so far had to come from somewhere. Company data usually lives in several places, such as JSON files, databases and the subscriber system's **API**, a web address that returns data when your code requests it. This week you pull Peakview Streaming data from all three and then clean it.

For this class, every source is saved as a file on the course GitHub, so everyone gets the same data and nothing can go down during class. Your code works exactly as it would with a live system, because you still request a web address and get data back.

## 1. What JSON looks like

Run the cell below to download the region file as text. Section 3 explains how `requests.get` works.

In [ ]:
import json
import requests
import pandas as pd

JSON_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/json/peakview_regions.json"
regions_text = requests.get(JSON_URL, timeout=10).text
print(regions_text[:500])

**1a.** Turn `regions_text` into Python with `json.loads` and store it as `regions`. Print the type of `regions`, the type of its first item, and how many regions there are. Then loop through `regions` and print each region's name, its city from the nested `office`, and how many `languages` it lists.

In [ ]:
# Your code here


**1b.** Use list comprehensions to list the regions whose `languages` include `"Spanish"`, and the regions whose `regional_manager` is `null` in the file. What does `null` become in Python?

In [ ]:
# Your code here


*Your answer here.*

**1c.** Why would `languages` be awkward to store in a CSV file?

*Your answer here.*

## 2. Query a SQLite database from pandas

Run the cell below to save the Peakview Streaming database into Colab and connect to it.

In [ ]:
import sqlite3

DB_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/sqlite/peakview.db"
with open("peakview.db", "wb") as f:
    f.write(requests.get(DB_URL, timeout=10).content)

conn = sqlite3.connect("peakview.db")

**2a.** List the database's tables, then show the first five rows of each one.

In [ ]:
# Your code here


**2b.** Write one SQL query that returns each region with its number of subscribers and its number of cancellations, using a `LEFT JOIN` from `subscribers` to `cancellations`. Read the result into a DataFrame named `by_region`. Then, in pandas, add a `cancel_rate` column and sort from highest to lowest.

In [ ]:
# Your code here


**2c.** Why does this query need a `LEFT JOIN` instead of a plain `JOIN`? And when would you do work in SQL before the data reaches pandas?

*Your answer here.*

## 3. Make one request

Run the cell below to store the address of the Peakview Streaming API.

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/peakview/"
first_url = BASE_URL + "subscribers_page_1.json"

**3a.** Request `first_url`. Print the status code, the keys of the response, `total_records`, `total_pages`, and the first subscriber in `results`.

In [ ]:
# Your code here


**3b.** Request `BASE_URL + "subscribers_page_99.json"`, a page that doesn't exist, and print its status code. Don't call `.json()` on it. What does the code mean, and why should you check the status code before calling `.json()`?

In [ ]:
# Your code here


*Your answer here.*

**API keys** are password-like strings that most real APIs require, to identify you and count your requests. Never type a key into a notebook cell, because notebooks get shared and pushed to GitHub, where automated tools scan for leaked keys. In Colab, save the key in the **Secrets** panel, the key icon in the left sidebar, and read it in code with `from google.colab import userdata` and then `api_key = userdata.get("MY_API_KEY")`. If a key does leak, revoke it with the provider and get a new one.

## 4. Get every page

**4a.** Collect every subscriber record from every page into a list called `records`. Print how many records you collected and compare it with `total_records` from page 1.

In [ ]:
# Your code here


**4b.** Why pause between requests? What could happen to a script that sends thousands of requests as fast as it can?

*Your answer here.*

## 5. Turn JSON into a DataFrame

**5a.** Turn `records` into a DataFrame named `subscribers` with `pd.json_normalize`. Rename these columns so they match the clean file you used in Modules 2 and 3, then print the shape and the column names.

| API name | New name |
| --- | --- |
| `signed_up` | `signup_date` |
| `plan.name` | `plan` |
| `plan.monthly_fee` | `monthly_fee` |
| `viewing.hours_last_30_days` | `hours_watched_last_30_days` |
| `viewing.hours_prior_30_days` | `hours_watched_prior_30_days` |
| `viewing.days_since_last_login` | `days_since_last_login` |

In [ ]:
# Your code here


## 6. Duplicate records

**6a.** Count the exact duplicate rows in `subscribers` and the repeated `subscriber_id` values. Drop the exact duplicates, then confirm that every `subscriber_id` is now unique and print the number of rows.

In [ ]:
# Your code here


## 7. Text that should be numbers or dates

**7a.** Count how many `signup_date` values contain a `/`, which marks the month/day/year format. Then convert `monthly_fee` to a number and convert `signup_date`, `next_login_date` and `cancel_date` to dates. Print the data types to check.

In [ ]:
# Your code here


## 8. Inconsistent categories and yes/no text

**8a.** Show `value_counts` for `region`, then fix it so each region has one spelling and show `value_counts` again. Then convert `autopay` to `True`/`False`.

In [ ]:
# Your code here


## 9. Placeholder codes and impossible values

**9a.** Count the subscribers whose `age` is below 0 or above 110, then set those ages to missing.

In [ ]:
# Your code here


**9b.** A subscriber can't log in after cancelling. Count the subscribers whose `next_login_date` is after their `cancel_date`, then set those `next_login_date` values to missing. Why is it safer to blank the login date than to delete the subscriber?

In [ ]:
# Your code here


*Your answer here.*

## 10. Check your work

Run the cell below. It loads the clean file from Modules 2 and 3 and compares it with your cleaned data, column by column. If your cleaning matches, every column shows 0 differences except `age`, where the ages you set to missing differ on purpose. If another column shows differences, go back to the section that cleans it.

In [ ]:
clean = pd.read_csv("https://raw.githubusercontent.com/skous2/is455-w2027/main/data/peakview_subscribers.csv", parse_dates=["signup_date", "next_login_date", "cancel_date"])
print(f"Your rows: {len(subscribers)}   Clean file rows: {len(clean)}")

if len(subscribers) == len(clean):
    mine = subscribers.sort_values("subscriber_id").reset_index(drop=True)
    for col in clean.columns:
        same = (mine[col] == clean[col]) | (mine[col].isna() & clean[col].isna())
        print(f"{col:30} {(~same).sum():>4} rows differ")

## Answer check

Fill in these values from your work above. They are graded on completion, and they show the instructor which ideas need more time on Thursday.

In [ ]:
# Records collected from every page, from 4a
answer_records_fetched = None

# Exact duplicate rows removed, from 6a
answer_duplicates_removed = None

# signup_date values containing "/", from 7a
answer_slash_dates = None

# Ages set to missing, from 9a
answer_ages_set_missing = None

# Logins after cancelling, from 9b
answer_logins_after_cancel = None

# Region with the highest cancel rate, from 2b. Use text, such as "Pacific".
answer_highest_cancel_region = None


## Before Thursday

**What is still confusing after this worksheet?** Be specific. Your answer decides what Thursday's class reteaches.

*Your answer here.*